# Validating torch.hub as the entry point for the model zoo

Proof of concept for distributing the 36 models published at [zenodo.org/records/19301058](https://zenodo.org/records/19301058) through `torch.hub`, with the architectures living in [Minerva](https://github.com/discovery-unicamp/Minerva) rather than in this repository.

**Files**

| File | Role |
|---|---|
| `hubconf.py` | torch.hub entrypoints. No model is hardcoded, it only reads the catalog |
| `models.yaml` | Catalog: 6 encoder architectures, 36 models, Zenodo URLs for both roles |
| `evaluate_daghar.py` | Evaluate one model on a DAGHAR test split |
| `validate_all.py` | Sweep the whole catalog against Table III of the paper |

**Questions this notebook answers**

1. Does torch.hub install the dependencies for you? (test 1)
2. Does it work when the model class lives outside the repository? (tests 3 and 5)
3. Can you list the models without downloading anything? (test 2)
4. Can it load the real Zenodo checkpoints? (test 5)
5. Do dynamically created entrypoints show up in the listing? (test 2)
6. Do the loaded models reproduce the published accuracies? (tests 7 and 8)

## 0. Setup

```bash
# main environment
python -m venv .venv-hubtest
.venv-hubtest/bin/pip install minerva==0.3.10b0 jupyter

# second environment WITHOUT Minerva, used by test 1
python -m venv .venv-hub-nominerva
.venv-hub-nominerva/bin/pip install torch --index-url https://download.pytorch.org/whl/cpu
.venv-hub-nominerva/bin/pip install pyyaml

# data, from the repository root (tests 7 and 8 only)
./download_data.sh
```

Use `./download_data.sh`. It fetches `standardized_view.zip` and leaves the datasets under `shared_data/daghar/standardized_view/<Dataset>` with 60-sample windows.

Do not substitute `prepare_data.py --only_daghar` for it: that path queries the Zenodo API and downloads every file in the record, including `baseline_view.zip`, extracting both into the same directory. The top-level folders then hold 150-sample windows and `Reshape((6, 60))` fails.

In [1]:
import importlib
import importlib.util
import subprocess
import sys
from pathlib import Path

import torch

REPO_DIR = Path.cwd()
GITHUB_REPO = "gustavo-luz/test_torch_hub"
PY_NO_MINERVA = Path("../.venv-hub-nominerva/bin/python").resolve()
DATA_ROOT = Path("../shared_data/daghar/standardized_view").resolve()

print("python             :", sys.version.split()[0])
print("torch              :", torch.__version__)
print("minerva installed  :", importlib.util.find_spec("minerva") is not None)
print("hubconf.py         :", (REPO_DIR / "hubconf.py").exists())
print("models.yaml        :", (REPO_DIR / "models.yaml").exists())
print("env without minerva:", PY_NO_MINERVA.exists())
print("DAGHAR data        :", DATA_ROOT.is_dir())
print("torch.hub cache    :", torch.hub.get_dir())

python             : 3.11.7
torch              : 2.13.0+cu130
minerva installed  : True
hubconf.py         : True
models.yaml        : True
env without minerva: True
DAGHAR data        : True
torch.hub cache    : /home/gustavo-luz/.cache/torch/hub


## Test 1. Without Minerva installed

Runs in a separate environment holding only `torch` and `pyyaml`. The question: does torch.hub resolve the dependency itself, that is, does it run `pip install`?

`hubconf.py` declares `dependencies = ["torch", "minerva", "yaml"]`, so we expect a clear error naming what is missing.

In [2]:
code = '''
import importlib.util as u, torch
print("minerva installed?", u.find_spec("minerva") is not None)
print("yaml installed?   ", u.find_spec("yaml") is not None)
try:
    m = torch.hub.load(REPO, "cnnpff", source="local")
    print("loaded:", type(m).__name__)
except Exception as e:
    print(f"{type(e).__name__}: {e}")
'''.replace("REPO", repr(str(REPO_DIR)))

if PY_NO_MINERVA.exists():
    r = subprocess.run([str(PY_NO_MINERVA), "-c", code], capture_output=True, text=True)
    print(r.stdout or r.stderr[-2000:])
else:
    print("environment without Minerva not found, see the setup section")

minerva installed? False
yaml installed?    True
RuntimeError: Missing dependencies: minerva



**Observed**

```
minerva installed? False
yaml installed?    True
RuntimeError: Missing dependencies: minerva
```

**Conclusion**: `dependencies` is a plain `importlib.util.find_spec` check. torch.hub **installs nothing** and knows nothing about versions: an incompatible Minerva passes the check and only fails later, at `load_state_dict`.

Version pinning therefore has to come from a pip package. A `hubconf.py` on its own cannot guarantee compatibility.

## Test 2. Listing the available models

Two paths: local (importing `hubconf.py` directly) and GitHub (`torch.hub.list`).

Note that `torch.hub.list()` does **not** accept `source="local"` (only `torch.hub.load` does), so listing without network access means importing the module.

In [3]:
sys.path.insert(0, str(REPO_DIR))
hubconf = importlib.import_module("hubconf")

entrypoints = [
    n for n in dir(hubconf) if not n.startswith("_") and callable(getattr(hubconf, n))
]
print(f"{len(entrypoints)} entrypoints in hubconf.py\n")
print("bare encoders:", [e for e in entrypoints if "_" not in e])
print("zoo models   :", [e for e in entrypoints if e.count("_") == 2][:5], "...")

# only 5 functions are written by hand, the rest is generated from models.yaml
print("\ndocstring of a generated entrypoint:")
print(" ", hubconf.lfr_ts2vec_ms.__doc__)

45 entrypoints in hubconf.py

bare encoders: ['cnnpff', 'harsccencoder', 'imutransformer', 'resnetse5', 'rnn', 'ts2vec', 'tstcc']
zoo models   : ['diet_resnetse5_kh', 'diet_resnetse5_uci', 'lfr_imutransformer_kh', 'lfr_imutransformer_rwthigh', 'lfr_imutransformer_rwwaist'] ...

docstring of a generated entrypoint:
  LFR + ts2vec trained on MotionSense (paper accuracy: 97.5%). role='pretrained'|'finetuned', head=True for the classifier.


In [4]:
# the catalog with a filter, no weights downloaded
for key, info in hubconf.list_models(dataset="ms").items():
    print(f"{key:<28} {info['dataset']:<16} acc={info['accuracy']}%  {info['roles']}")

lfr_resnetse5_ms             MotionSense      acc=93.8%  ['finetuned', 'pretrained']
lfr_ts2vec_ms                MotionSense      acc=97.5%  ['finetuned', 'pretrained']
lfr_tstcc_ms                 MotionSense      acc=92.8%  ['finetuned', 'pretrained']
tfc_cnnpff_ms                MotionSense      acc=95.0%  ['finetuned', 'pretrained']
tfc_imutransformer_ms        MotionSense      acc=89.7%  ['finetuned', 'pretrained']
tfc_rnn_ms                   MotionSense      acc=92.0%  ['finetuned', 'pretrained']


In [5]:
# through GitHub: requires hubconf.py and models.yaml to be published there
try:
    eps = torch.hub.list(GITHUB_REPO, trust_repo=True)
    print(f"{len(eps)} entrypoints via GitHub")
    print(eps[:10])
except Exception as e:
    print(f"{type(e).__name__}: {e}")

45 entrypoints via GitHub
['cnnpff', 'diet_resnetse5_kh', 'diet_resnetse5_uci', 'get_model', 'harsccencoder', 'imutransformer', 'lfr_imutransformer_kh', 'lfr_imutransformer_rwthigh', 'lfr_imutransformer_rwwaist', 'lfr_resnetse5_ms']


Using cache found in /home/gustavo-luz/.cache/torch/hub/gustavo-luz_test_torch_hub_main


## Test 3. Loading an architecture with no weights

`cnnpff` is Minerva's CNN-PFF encoder, randomly initialised here. This is the core question: the class lives in Minerva, not in this repository.

In [6]:
model = torch.hub.load(str(REPO_DIR), "cnnpff", source="local")

print(type(model).__module__ + "." + type(model).__name__)
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")

model.eval()
with torch.no_grad():
    y = model(torch.randn(4, 6, 60))  # DAGHAR window: 6 IMU channels, 60 samples
print("input (4, 6, 60) -> output", tuple(y.shape))

/home/gustavo-luz/code/hiaac/OFFICIAL_PR/benchmarking-encoders-ssl-har/.venv-hubtest/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


minerva.models.nets.time_series.cnns.CNN_PF_Backbone
parameters: 46,624
input (4, 6, 60) -> output (4, 768)


## Test 4. Loading from GitHub

Here torch.hub downloads the repository zipball into `~/.cache/torch/hub`, imports `hubconf.py` and calls the function by name. This is the flow an end user would see.

In [7]:
try:
    model_gh = torch.hub.load(GITHUB_REPO, "cnnpff", trust_repo=True)
    print("loaded from GitHub:", type(model_gh).__name__)
except Exception as e:
    print(f"{type(e).__name__}: {e}")

loaded from GitHub: CNN_PF_Backbone


Using cache found in /home/gustavo-luz/.cache/torch/hub/gustavo-luz_test_torch_hub_main


In [8]:
# the exact flow an end user gets: one line, nothing cloned by hand
for name, kwargs, label in [
    ("cnnpff", {}, "architecture only"),
    ("lfr_ts2vec_ms", {"role": "pretrained"}, "SSL backbone"),
    ("lfr_ts2vec_ms", {"role": "finetuned", "head": True}, "full classifier"),
    ("tfc_ts2vec_kh", {"role": "pretrained"}, "TFC path"),
]:
    m = torch.hub.load(GITHUB_REPO, name, trust_repo=True, **kwargs)
    m.eval()
    with torch.no_grad():
        y = m(torch.randn(2, 6, 60))
    print(f"{name:<16} {label:<18} {type(m).__name__:<22} -> {tuple(y.shape)}")

Using cache found in /home/gustavo-luz/.cache/torch/hub/gustavo-luz_test_torch_hub_main


cnnpff           architecture only  CNN_PF_Backbone        -> (2, 768)


Using cache found in /home/gustavo-luz/.cache/torch/hub/gustavo-luz_test_torch_hub_main


lfr_ts2vec_ms    SSL backbone       TSEncoder              -> (2, 60, 320)


Using cache found in /home/gustavo-luz/.cache/torch/hub/gustavo-luz_test_torch_hub_main


lfr_ts2vec_ms    full classifier    SimpleSupervisedModel  -> (2, 6)


Using cache found in /home/gustavo-luz/.cache/torch/hub/gustavo-luz_test_torch_hub_main


tfc_ts2vec_kh    TFC path           TFC_Backbone           -> (2, 256)


**Observed**, against `gustavo-luz/test_torch_hub`:

```
cnnpff           architecture only  CNN_PF_Backbone        -> (2, 768)
lfr_ts2vec_ms    SSL backbone       TSEncoder              -> (2, 60, 320)
lfr_ts2vec_ms    full classifier    SimpleSupervisedModel  -> (2, 6)
tfc_ts2vec_kh    TFC path           TFC_Backbone           -> (2, 256)
```

torch.hub caches the repository zipball under `~/.cache/torch/hub/<owner>_<repo>_<branch>` and reuses it, so only the first call hits the network. Pass `force_reload=True` after pushing a new commit, otherwise the stale copy is used.

A model loaded this way, straight from GitHub, scores **97.46%** on the MotionSense test split, the same number as the local path and 0.04 pp from the paper. The whole chain works: GitHub for the code, Zenodo for the weights, Minerva for the architectures.

## Test 5. Real weights from Zenodo

The step that matters: the entrypoint builds the right architecture, downloads the `.ckpt` with `torch.hub.load_state_dict_from_url` and loads the backbone weights.

Two details handled in `hubconf.py`:

- `weights_only=False`, because the checkpoint comes from Lightning and torch >= 2.6 defaults to `weights_only=True`;
- filtering the `backbone.` prefix, which is what `FromPretrained` does in the official notebook.

In [9]:
# LFR + TS2Vec on MotionSense: best result in the paper, 97.5%
backbone = torch.hub.load(
    str(REPO_DIR), "lfr_ts2vec_ms", role="pretrained", source="local"
)
backbone.eval()
with torch.no_grad():
    out = backbone(torch.randn(4, 6, 60))

print(type(backbone).__name__, f"{sum(p.numel() for p in backbone.parameters()):,} params")
print("backbone output:", tuple(out.shape))

TSEncoder 637,568 params
backbone output: (4, 60, 320)


In [10]:
# sanity check: the loaded weights really differ from a fresh initialisation
random_init = torch.hub.load(str(REPO_DIR), "lfr_ts2vec_ms", source="local")

a = next(backbone.parameters()).flatten()[:4]
b = next(random_init.parameters()).flatten()[:4]
print("pretrained:", [round(v, 4) for v in a.tolist()])
print("random    :", [round(v, 4) for v in b.tolist()])
print("identical?", torch.allclose(a, b))

pretrained: [-0.2663, -0.3567, -0.1304, 0.1998]
random    : [0.0296, -0.2724, -0.0066, -0.0808]
identical? False


In [11]:
# full classifier: finetuned backbone + MLP head, 6 activities
clf = torch.hub.load(
    str(REPO_DIR), "lfr_ts2vec_ms", role="finetuned", head=True, source="local"
)
clf.eval()
with torch.no_grad():
    logits = clf(torch.randn(4, 6, 60))
print("logits:", tuple(logits.shape), "(batch, 6 activities)")

logits: (4, 6) (batch, 6 activities)


## Test 6. Building the whole catalog

Builds **all** 36 architectures without downloading weights, to confirm that `models.yaml` stays consistent with the installed Minerva. This is the seed of a weekly CI job.

In [12]:
ok, failed = [], []

for key in hubconf.list_models():
    try:
        m = hubconf.get_model(key)
        ok.append((key, sum(p.numel() for p in m.parameters())))
    except Exception as e:
        failed.append((key, f"{type(e).__name__}: {str(e)[:100]}"))

print(f"built: {len(ok)}/36")
for key, n in ok[:5]:
    print(f"  {key:<28} {n:>12,} params")
print("  ...")

for key, err in failed:
    print(f"  FAILED {key:<28} {err}")

/home/gustavo-luz/code/hiaac/OFFICIAL_PR/benchmarking-encoders-ssl-har/.venv-hubtest/lib/python3.11/site-packages/minerva/models/nets/time_series/imu_transformer.py:66: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.transformer_encoder = TransformerEncoder(


built: 36/36
  diet_resnetse5_kh                 126,912 params
  diet_resnetse5_uci                126,912 params
  lfr_imutransformer_kh             217,856 params
  lfr_imutransformer_rwthigh        217,856 params
  lfr_imutransformer_rwwaist        217,856 params
  ...


## Test 7. Inference on real windows

With the classifier loaded, predict activities for windows of the DAGHAR test split. This is a spot check, the full evaluation is test 8.


In [13]:
import pandas as pd

ACTIVITIES = {0: "sit", 1: "stand", 2: "walk", 3: "stair up", 4: "stair down", 5: "run"}
CHANNELS = ["accel-x", "accel-y", "accel-z", "gyro-x", "gyro-y", "gyro-z"]

df = pd.read_csv(DATA_ROOT / "MotionSense" / "test.csv")
cols = [c for ch in CHANNELS for c in df.columns if c.startswith(ch + "-")]
print("windows in the test split:", len(df))
print("channels x samples       :", len(cols) // 60, "x 60")

windows in the test split: 1062
channels x samples       : 6 x 60


In [14]:
# spot check on a small batch, fixed seed so the cell is reproducible
batch = df.sample(12, random_state=0)
x = torch.tensor(batch[cols].values.astype("float32")).reshape(-1, 6, 60)
y = batch["standard activity code"].to_numpy()

with torch.no_grad():
    probs = clf(x).softmax(1)
pred = probs.argmax(1).numpy()

print(f"{'true':>11}  {'predicted':>11}  {'confidence':>10}")
for t, p_, c in zip(y, pred, probs.max(1).values):
    mark = "" if t == p_ else "   <- miss"
    print(f"{ACTIVITIES[t]:>11}  {ACTIVITIES[p_]:>11}  {c:>9.1%}{mark}")
print(f"\n{(y == pred).sum()}/{len(y)} correct in this batch, "
      f"97.46% over the full test split (test 8)")

       true    predicted  confidence
        sit          sit     100.0%
        run          run     100.0%
      stand        stand      53.8%
        run          run     100.0%
        run          run     100.0%
   stair up     stair up      99.7%
       walk         walk     100.0%
        run          run     100.0%
      stand        stand     100.0%
       walk         walk     100.0%
       walk         walk     100.0%
 stair down   stair down      99.9%

12/12 correct in this batch, 97.46% over the full test split (test 8)


In [15]:
# the class probabilities behind one of those predictions
row = batch.iloc[0]
one = torch.tensor(row[cols].values.astype("float32")).reshape(1, 6, 60)
with torch.no_grad():
    p_one = clf(one).softmax(1)[0]

print(f"true: {ACTIVITIES[int(row['standard activity code'])]}\n")
for i, v in enumerate(p_one):
    print(f"  {ACTIVITIES[i]:>11}: {'#' * int(v * 40):<40} {v:.1%}")

true: sit

          sit: ######################################## 100.0%
        stand:                                          0.0%
         walk:                                          0.0%
     stair up:                                          0.0%
   stair down:                                          0.0%
          run:                                          0.0%


## Test 8. Does it reproduce the paper?

`evaluate_daghar.py` evaluates one model on a full test split with the same data pipeline as the official notebook: `CSVReader` over the 6 IMU columns, `Reshape((6, 60))`, label in `standard activity code`.

```bash
python evaluate_daghar.py                    # lfr_ts2vec_ms on MotionSense
python evaluate_daghar.py tfc_ts2vec_kh      # another model
python evaluate_daghar.py lfr_ts2vec_ms uci  # cross-dataset transfer
```

In [16]:
r = subprocess.run(
    [sys.executable, "evaluate_daghar.py", "lfr_ts2vec_ms"],
    capture_output=True,
    text=True,
)
print(r.stdout)

model     : lfr_ts2vec_ms  (LFR + ts2vec)
evaluating: MotionSense  (test.csv)
paper     : 97.5% on its training dataset

accuracy  : 97.5%
vs paper  : identical to the paper

confusion matrix (row = true, column = predicted)
                     sit       stand        walk    stair up  stair down         run
         sit         176           1           0           0           0           0
       stand           7         170           0           0           0           0
        walk           0           0         174           3           0           0
    stair up           0           0           3         168           6           0
  stair down           0           0           3           0         172           2
         run           0           0           0           0           2         175



### All 36 models

`validate_all.py` runs the same evaluation across the whole catalog and compares each model with Table III.

```bash
python validate_all.py               # all 36, around 2 min once the checkpoints are cached
python validate_all.py --ssl tfc     # one technique
python validate_all.py --dataset ms  # one dataset
```

It writes `validation_results.csv`, read below.

In [17]:
results = pd.read_csv("validation_results.csv")

# the paper prints one decimal, so 0.1 is a single step of the last digit
print(f"models validated : {len(results)}")
print(f"within 0.1 pp    : {(results['delta'].abs() <= 0.1).sum()}/{len(results)}")
print(f"exact match      : {(results['delta'] == 0).sum()}/{len(results)}")
print(f"largest deviation: {results['delta'].abs().max():.1f} pp")

results.sort_values("paper", ascending=False)[
    ["model", "dataset", "paper", "measured", "delta"]
]

models validated : 36
within 0.1 pp    : 36/36
exact match      : 33/36
largest deviation: 0.1 pp


,model,dataset,paper,measured,delta
6,lfr_ts2vec_ms,MotionSense,97.5,97.5,0.0
13,tfc_cnnpff_uci,UCI-HAR,96.2,96.2,0.0
30,tfc_ts2vec_uci,UCI-HAR,96.1,96.1,0.0
1,diet_resnetse5_uci,UCI-HAR,95.9,95.8,-0.1
10,tfc_cnnpff_ms,MotionSense,95.0,95.1,0.1
25,tfc_rnn_uci,UCI-HAR,94.9,94.9,0.0
34,tfc_tstcc_uci,UCI-HAR,94.3,94.3,0.0
5,lfr_resnetse5_ms,MotionSense,93.8,93.8,0.0
7,lfr_tstcc_ms,MotionSense,92.8,92.8,0.0
16,tfc_imutransformer_uci,UCI-HAR,92.3,92.3,0.0


In [18]:
# where the deviations sit, by technique and by dataset
print(results.groupby("ssl")["delta"].agg(["count", "mean", "min", "max"]).round(3))
print()
print(results.groupby("dataset")["delta"].agg(["count", "mean", "min", "max"]).round(3))

      count   mean  min  max
ssl                         
diet      2 -0.050 -0.1  0.0
lfr       7  0.000  0.0  0.0
tfc      27  0.007  0.0  0.1

                 count   mean  min  max
dataset                                
KuHar                6  0.000  0.0  0.0
MotionSense          6  0.017  0.0  0.1
RealWorld Thigh      6  0.017  0.0  0.1
RealWorld Waist      6  0.000  0.0  0.0
UCI-HAR              6 -0.017 -0.1  0.0
WISDM                6  0.000  0.0  0.0


### Validation result

All 36 finetuned checkpoints, loaded through torch.hub and evaluated on the test split of their own dataset. Accuracy is compared at one decimal, the precision of Table III, where a deviation of 0.1 is a single step of the last printed digit.

| | |
|---|---|
| Models validated | **36 / 36** |
| Within 0.1 pp of the published accuracy | **36 / 36** |
| Exact match | **33 / 36** |
| Total runtime | 4.9 min on CPU, most of it downloading two 84 MB checkpoints |

The three that are one step off: `diet_resnetse5_uci` (95.8 against 95.9), `tfc_cnnpff_ms` (95.1 against 95.0) and `tfc_ts2vec_rwthigh` (81.3 against 81.2).

That confirms three things at once: the architectures rebuilt from `models.yaml` match the ones that produced the checkpoints, the `backbone.` and `fc.` key filtering is right, and the head dimensions per SSL technique are correct.


## Findings

Measured with torch 2.13.0 and minerva 0.3.10b0 in a clean environment.

| Question | Answer |
|---|---|
| Does `dependencies` install anything? | **No**. It only checks with `find_spec` and raises `RuntimeError: Missing dependencies: minerva` |
| Does a class outside the repository work? | **Yes**. Minerva's `CNN_PF_Backbone` loaded through `torch.hub.load`, output `(4, 768)` |
| Does `torch.hub.list` work locally? | **No**. `list()` has no `source` parameter, only `load()` does. Import `hubconf` directly instead |
| Do generated entrypoints show up? | **Yes**. 45 entrypoints over GitHub too, 42 of them generated from `models.yaml` |
| Do the Zenodo weights load? | **Yes**, cached under `~/.cache/torch/hub/checkpoints`, prefix filtered, `strict=True` |
| Does a separate catalog file work? | **Yes**. All 36 architectures build from `models.yaml` |
| Does it reproduce the paper? | **Yes**. 36/36 within 0.1 pp, 33 of them exact |

### Traps worth knowing

1. **`dependencies` takes module names, not package names**: `yaml`, not `PyYAML`.
2. **`torch.hub.list` only looks at GitHub**, it does not accept `source="local"`.
3. **Every public callable becomes an entrypoint**: a plain `from pathlib import Path` gets listed as if it were a model. Import with a leading underscore.
4. **`minerva.__version__` lies**: pip installed `0.3.10b0` while the attribute reports `0.3.8-beta`. A runtime compatibility check has to use `importlib.metadata.version`.
5. **Lightning checkpoints**: `weights_only=True` happens to work on these files, but `weights_only=False` is what always loads on torch >= 2.6.
6. **Use `./download_data.sh`** for the data. `prepare_data.py --only_daghar` pulls the whole Zenodo record and mixes the baseline view (150 samples) into the standardized one (60), while still printing "All 6 DAGHAR datasets found".

### What this means for the proposal

- `hubconf.py` solves **ergonomics** (one line to load a model) but not **version compatibility**: it neither installs nor checks the Minerva version.
- The three-layer proposal still holds: a pip package carrying the version pin as the base, `hubconf.py` as a shortcut, weights served by URL and kept out of git.
- The `models.yaml` catalog already removes the hand-copied `build_backbone` and `_HEAD_INPUT` tables from the official notebook. The 36/36 validation is the evidence that the generated architectures are equivalent.

### Validated over GitHub

With `hubconf.py`, `models.yaml`, `README.md` and `evaluate_daghar.py` pushed to `gustavo-luz/test_torch_hub`, the remote path behaves like the local one: `torch.hub.list` returns the 45 entrypoints, `torch.hub.help` prints the generated docstring, and a classifier loaded from GitHub reproduces 97.46% on MotionSense.

Two things to watch when the repository moves to `H-IAAC/benchmarking-encoders-ssl-har`:

1. The cache key is `<owner>_<repo>_<branch>`, so users who tried the personal repo keep a stale copy until `force_reload=True` or a manual cache wipe.
2. `hubconf.py` has to sit at the repository root. In the paper repo that means the root, not a subdirectory, which is worth deciding before publishing.
### Still open

| Item | Why it matters |
|---|---|
| `harsccencoder` is broken in the published version | It raises `NameError: HARSCnnEncoder is not defined`, the import was dropped from the function body. Fixed locally, not pushed |
| `test_torch_hub.ipynb`, `validate_all.py` and `validation_results.csv` are not published | The repository carries the code but not the evidence, so nobody can rerun the validation from a clone |
| `.gitignore` just added | `.venv-hubtest/` sits inside the repository at 1.1 GB, one `git add -A` away from being committed |
| Move to `H-IAAC/benchmarking-encoders-ssl-har` | `hubconf.py` has to live at the repository root, and the torch.hub cache key changes with the owner |
| Weights still come from Zenodo only | No mirror yet, so a Zenodo outage takes the zoo down. See the Hugging Face proposal |
| No CI | Nothing catches a Minerva release that breaks the catalog. `validate_all.py` is the job to schedule |
